In [1]:
import pandas as pd
import altair as alt
import numpy as np

In [2]:
defunciones = pd.read_parquet("data/processed/defunciones_agregado_2005_2024.parquet")

In [3]:
defunciones["anio_ocur"] = pd.to_numeric(
    defunciones["anio_ocur"],
    errors="coerce"
).astype("Int64")

In [4]:
defunciones = defunciones[defunciones["anio_ocur"] >= 2005]

In [5]:
defunciones = defunciones.drop(columns=["lista_mex", "prefijo", "lista1", "capitulo", "grupo", "sexo", "edad_agru", "provisional", "poblacion", "tasa_100k"], axis=1)

In [6]:
poblacion = pd.read_csv("data/processed/poblacion_entidades.csv", index_col=0)

In [7]:
poblacion["ANIO"] = pd.to_numeric(
    poblacion["ANIO"],
    errors="coerce"
).astype("Int64")

In [8]:
poblacion = poblacion[poblacion["ANIO"] >= 2005]

In [9]:
poblacion.head()

,ANIO,ENTIDAD,CVE_GEO,EDAD,SEXO,POBLACION,ENTIDAD_FEDERATIVA,FECHA
RENGLON,,,,,,,,
258721,2005,Aguascalientes,1,0,Hombres,12856,Aguascalientes,2005-01-01
258722,2005,Aguascalientes,1,0,Mujeres,12423,Aguascalientes,2005-01-01
258723,2005,Aguascalientes,1,1,Hombres,12925,Aguascalientes,2005-01-01
258724,2005,Aguascalientes,1,1,Mujeres,12516,Aguascalientes,2005-01-01
258725,2005,Aguascalientes,1,2,Hombres,13076,Aguascalientes,2005-01-01


In [10]:
poblacion_agrupada = (
    poblacion
    .groupby(["ANIO", "EDAD", "SEXO"], as_index=False)["POBLACION"]
    .sum()
)

poblacion_agrupada.head()

,ANIO,EDAD,SEXO,POBLACION
0,2005,0,Hombres,1174839
1,2005,0,Mujeres,1134996
2,2005,1,Hombres,1170941
3,2005,1,Mujeres,1134099
4,2005,2,Hombres,1174596


In [11]:
poblacion_por_anio = (
    poblacion_agrupada
    .groupby("ANIO", as_index=False)["POBLACION"]
    .sum()
)

poblacion_por_anio.head()

,ANIO,POBLACION
0,2005,106370948
1,2006,107885646
2,2007,109538665
3,2008,111275915
4,2009,113030218


In [12]:
defunciones = (
    defunciones
    .merge(
        poblacion_por_anio,
        how="left",
        left_on="anio_ocur",
        right_on="ANIO"
    )
    .drop(columns="ANIO")
)

defunciones.head()

,anio_ocur,lista_mex_desc,lista_mex_desc_anio_fuente,sexo_desc,edad_agru_desc,defunciones,POBLACION
0,2005,Fiebre tifoidea,2005,Hombre,Menores de un año,2,106370948
1,2005,Fiebre tifoidea,2005,Hombre,De 10 a 14 años,3,106370948
2,2005,Fiebre tifoidea,2005,Hombre,De 20 a 24 años,2,106370948
3,2005,Fiebre tifoidea,2005,Hombre,De 25 a 29 años,1,106370948
4,2005,Fiebre tifoidea,2005,Hombre,De 30 a 34 años,2,106370948


In [13]:
defunciones_por_anio = (
    defunciones.groupby("anio_ocur", as_index=False)["defunciones"]
    .sum()
)

alt.Chart(defunciones_por_anio).mark_bar(size=20).encode(
    y=alt.Y("defunciones:Q", title="Defunciones"),
    x=alt.X("anio_ocur:Q", title="Año", sort="ascending"),
    tooltip=[
        alt.Tooltip("anio_ocur:N", title="Año"),
        alt.Tooltip("defunciones:Q", title="Defunciones", format=",")
    ]
).properties(
    width=700,
    height=400,
    title="Defunciones por año"
).transform_filter(
    alt.datum.anio_ocur >= 2005
)

alt.Chart(...)

In [14]:
defunciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208691 entries, 0 to 208690
Data columns (total 7 columns):
 #   Column                      Non-Null Count   Dtype 
---  ------                      --------------   ----- 
 0   anio_ocur                   208691 non-null  Int64 
 1   lista_mex_desc              208691 non-null  object
 2   lista_mex_desc_anio_fuente  208691 non-null  int16 
 3   sexo_desc                   208691 non-null  object
 4   edad_agru_desc              208691 non-null  object
 5   defunciones                 208691 non-null  int64 
 6   POBLACION                   208691 non-null  int64 
dtypes: Int64(1), int16(1), int64(2), object(3)
memory usage: 10.2+ MB


In [15]:
defunciones_total = (
    defunciones
    .groupby(
        ["anio_ocur", "lista_mex_desc", "POBLACION"],
        as_index=False,
        dropna=False
    )["defunciones"]
    .sum()
)

In [16]:
defunciones_total.head()

,anio_ocur,lista_mex_desc,POBLACION,defunciones
0,2005,Aborto espontáneo,106370948,2
1,2005,Accidente causado por maquinaria y por instrum...,106370948,170
2,2005,Accidente causado por proyectil de arma de fuego,106370948,276
3,2005,Accidente vascular encefálico agudo no especif...,106370948,4854
4,2005,Accidentes de ferrocarril,106370948,188


In [17]:
defunciones_total["tasa_100k"] = (
    defunciones_total["defunciones"]
    / defunciones_total["POBLACION"]
    * 100_000
)

defunciones_total.head()

,anio_ocur,lista_mex_desc,POBLACION,defunciones,tasa_100k
0,2005,Aborto espontáneo,106370948,2,0.001880
1,2005,Accidente causado por maquinaria y por instrum...,106370948,170,0.159818
2,2005,Accidente causado por proyectil de arma de fuego,106370948,276,0.259469
3,2005,Accidente vascular encefálico agudo no especif...,106370948,4854,4.563276
4,2005,Accidentes de ferrocarril,106370948,188,0.176740


In [18]:
defunciones_total

,anio_ocur,lista_mex_desc,POBLACION,defunciones,tasa_100k
0,2005,Aborto espontáneo,106370948,2,0.001880
1,2005,Accidente causado por maquinaria y por instrum...,106370948,170,0.159818
2,2005,Accidente causado por proyectil de arma de fuego,106370948,276,0.259469
3,2005,Accidente vascular encefálico agudo no especif...,106370948,4854,4.563276
4,2005,Accidentes de ferrocarril,106370948,188,0.176740
...,...,...,...,...,...
6286,2024,Tétanos neonatal,132274416,1,0.000756
6287,2024,Ulceras gástrica y duodenal,132274416,3077,2.326225
6288,2024,Vacunas COVID-19 que causan efectos adversos e...,132274416,1,0.000756
6289,2024,Varicela y herpes zoster,132274416,57,0.043092


In [19]:
diferencia_tasa = (
    defunciones_total[
        defunciones_total["anio_ocur"].isin([2005, 2024])
    ]
    .pivot(
        index="lista_mex_desc",
        columns="anio_ocur",
        values="tasa_100k"
    )
    .assign(
        diferencia_tasa_100k=lambda df: df[2024] - df[2005]
    )
    .reset_index()
    .loc[:, ["lista_mex_desc", "diferencia_tasa_100k"]]
)

diferencia_tasa.head()

anio_ocur,lista_mex_desc,diferencia_tasa_100k
0,Aborto espontáneo,0.000388
1,Aborto médico,NaN
2,Accidente causado por maquinaria y por instrum...,-0.026761
3,Accidente causado por proyectil de arma de fuego,0.113241
4,Accidente vascular encefálico agudo no especif...,1.155895


In [20]:
diferencia_tasa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   lista_mex_desc        330 non-null    object 
 1   diferencia_tasa_100k  294 non-null    float64
dtypes: float64(1), object(1)
memory usage: 5.3+ KB


In [21]:
diferencia_tasa["diferencia_tasa_100k"].isna().sum()

np.int64(36)

In [22]:
# filter rows with NaN values in the "diferencia_tasa_100k" column
nulos = diferencia_tasa[diferencia_tasa["diferencia_tasa_100k"].isna()]

In [23]:
nulos

anio_ocur,lista_mex_desc,diferencia_tasa_100k
1,Aborto médico,NaN
29,COVID-19,NaN
30,Carcinoma in situ del cuello del útero,NaN
31,Caries dental,NaN
35,Ceguera y disminución de la agudeza visual,NaN
43,"Cólico renal, no especificado",NaN
44,Deficiencia de vitamina A,NaN
45,Deformidades adquiridas de los miembros,NaN
46,Deformidades congénitas de la cadera,NaN
47,Deformidades congénitas de los pies,NaN


In [24]:
# Mayor diferencia (descending)
top_30_diferencia = (
    diferencia_tasa
    .dropna(subset=["diferencia_tasa_100k"])
    .nlargest(30, "diferencia_tasa_100k")
)

alt.Chart(top_30_diferencia).mark_bar().encode(
    x=alt.X(
        "diferencia_tasa_100k:Q",
        title="Diferencia de tasa por 100,000 habitantes",
        axis=alt.Axis(format=".2f")
    ),
    y=alt.Y(
        "lista_mex_desc:N",
        sort="-x",
        title="Causa de defunción"
    ),
    tooltip=[
        alt.Tooltip("lista_mex_desc:N", title="Causa"),
        alt.Tooltip(
            "diferencia_tasa_100k:Q",
            title="Diferencia",
            format=".3f"
        )
    ]
).properties(
    width=500,
    height=700,
    title="Top 30 causas con mayor diferencia en la tasa"
)

alt.Chart(...)

In [25]:
top_30_diferencia

anio_ocur,lista_mex_desc,diferencia_tasa_100k
117,Infarto agudo del miocardio,53.645783
51,Diabetes mellitus,19.586833
11,Agresiones (homicidios),14.837388
177,Neumonía,14.657964
201,Otras enfermedades del sistema urinario,5.273230
198,Otras enfermedades del hígado,4.569614
199,Otras enfermedades del sistema digestivo,4.546049
319,Tumores de comportamiento incierto o desconocido,3.827898
146,Las demás enfermedades hipertensivas,3.571780
114,Hipertensión esencial,3.152809


In [26]:
top_30_lista_mex_desc = top_30_diferencia["lista_mex_desc"].tolist()

top_30_lista_mex_desc

['Infarto agudo del miocardio',
 'Diabetes mellitus',
 'Agresiones (homicidios)',
 'Neumonía',
 'Otras enfermedades del sistema urinario',
 'Otras enfermedades del hígado',
 'Otras enfermedades del sistema digestivo',
 'Tumores de comportamiento incierto o desconocido',
 'Las demás enfermedades hipertensivas',
 'Hipertensión esencial',
 'Insuficiencia renal',
 'Las demás enfermedades endocrinas y metabólicas',
 'Infecciones de la piel y del tejido subcutáneo',
 'Septicemia',
 'Tumor maligno del colon',
 'Lesiones autoinfligidas intencionalmente',
 'Enfermedad cardíaca hipertensiva',
 'Otras enfermedades de los intestinos y del peritoneo',
 'Tumor maligno de la mama',
 'Las demás enfermedades del aparato respiratorio',
 'Trastornos de la conducción y arritmias cardíacas',
 'Síndrome nefrítico agudo y síndrome nefrítico rápidamente progresivo',
 'Ileo paralítico y obstrucción intestinal sin hernia',
 'Accidente vascular encefálico agudo no especificado como hemorrágico o isquémico',
 'Tu

In [28]:
series_top_30 = defunciones_total[
    defunciones_total["lista_mex_desc"].isin(top_30_lista_mex_desc)
]

alt.Chart(series_top_30).mark_line(point=True).encode(
    x=alt.X(
        "anio_ocur:Q",
        title="Año",
        scale=alt.Scale(domain=[2005, 2024])
    ),
    y=alt.Y(
        "tasa_100k:Q",
        title="Tasa por 100,000 habitantes"
    ),
    color=alt.Color(
        "lista_mex_desc:N",
        title="Causa de defunción",
        legend=alt.Legend(columns=2)
    ),
    tooltip=[
        alt.Tooltip("anio_ocur:Q", title="Año"),
        alt.Tooltip("lista_mex_desc:N", title="Causa"),
        alt.Tooltip("tasa_100k:Q", title="Tasa", format=".2f")
    ]
).properties(
    width=800,
    height=500,
    title="Evolución de las tasas de defunción: 30 principales causas"
)

alt.Chart(...)

In [29]:
# Mayor diferencia (descending)
top_10_diferencia = (
    diferencia_tasa
    .dropna(subset=["diferencia_tasa_100k"])
    .nlargest(10, "diferencia_tasa_100k")
)

top_10_lista_mex_desc = top_10_diferencia["lista_mex_desc"].tolist()

top_10_lista_mex_desc

['Infarto agudo del miocardio',
 'Diabetes mellitus',
 'Agresiones (homicidios)',
 'Neumonía',
 'Otras enfermedades del sistema urinario',
 'Otras enfermedades del hígado',
 'Otras enfermedades del sistema digestivo',
 'Tumores de comportamiento incierto o desconocido',
 'Las demás enfermedades hipertensivas',
 'Hipertensión esencial']

In [30]:
series_top_10 = defunciones_total[
    defunciones_total["lista_mex_desc"].isin(top_10_lista_mex_desc)
]

alt.Chart(series_top_10).mark_line(point=True).encode(
    x=alt.X(
        "anio_ocur:Q",
        title="Año",
        scale=alt.Scale(domain=[2005, 2024])
    ),
    y=alt.Y(
        "tasa_100k:Q",
        title="Tasa por 100,000 habitantes"
    ),
    color=alt.Color(
        "lista_mex_desc:N",
        title="Causa de defunción",
        legend=alt.Legend(columns=2)
    ),
    tooltip=[
        alt.Tooltip("anio_ocur:Q", title="Año"),
        alt.Tooltip("lista_mex_desc:N", title="Causa"),
        alt.Tooltip("tasa_100k:Q", title="Tasa", format=".2f")
    ]
).properties(
    width=800,
    height=500,
    title="Evolución de las tasas de defunción: 10 principales causas"
)

alt.Chart(...)

In [31]:
series_top_10

,anio_ocur,lista_mex_desc,POBLACION,defunciones,tasa_100k
10,2005,Agresiones (homicidios),106370948,10003,9.403883
42,2005,Diabetes mellitus,106370948,67381,63.345304
100,2005,Hipertensión esencial,106370948,4068,3.824352
103,2005,Infarto agudo del miocardio,106370948,45008,42.312305
130,2005,Las demás enfermedades hipertensivas,106370948,3673,3.453010
...,...,...,...,...,...
6145,2024,Neumonía,132274416,35746,27.024122
6166,2024,Otras enfermedades del hígado,132274416,26565,20.083249
6167,2024,Otras enfermedades del sistema digestivo,132274416,10429,7.884367
6169,2024,Otras enfermedades del sistema urinario,132274416,8920,6.743557


In [32]:
alt.Chart(series_top_10).mark_line(point=True).encode(
    x=alt.X(
        "anio_ocur:Q",
        title="Año",
        scale=alt.Scale(domain=[2005, 2024])
    ),
    y=alt.Y(
        "defunciones:Q",
        title="Número de defunciones"
    ),
    color=alt.Color(
        "lista_mex_desc:N",
        title="Causa de defunción",
        legend=alt.Legend(columns=2)
    ),
    tooltip=[
        alt.Tooltip("anio_ocur:Q", title="Año"),
        alt.Tooltip("lista_mex_desc:N", title="Causa"),
        alt.Tooltip("defunciones:Q", title="Defunciones", format=".0f")
    ]
).properties(
    width=800,
    height=500,
    title="Evolución de las defunciones: 10 principales causas"
)

alt.Chart(...)

In [38]:
agresiones = defunciones_total[
    defunciones_total["lista_mex_desc"] == "Agresiones (homicidios)"
]

barras = alt.Chart(agresiones).mark_bar(color="#31CFFF", size=30).encode(
    x=alt.X("anio_ocur:Q", title="Año", axis=alt.Axis(format="d")),
    y=alt.Y(
        "defunciones:Q",
        title="Defunciones",
        axis=alt.Axis(titleColor="#4C78A8")
    ),
    tooltip=[
        alt.Tooltip("anio_ocur:Q", title="Año"),
        alt.Tooltip("defunciones:Q", title="Defunciones", format=",")
    ]
)

linea = alt.Chart(agresiones).mark_line(
    color="#7B7B7B",
    point=True
).encode(
    x="anio_ocur:Q",
    y=alt.Y(
        "tasa_100k:Q",
        title="Tasa por 100,000 habitantes",
        axis=alt.Axis(titleColor="#E45756")
    ),
    tooltip=[
        alt.Tooltip("anio_ocur:Q", title="Año"),
        alt.Tooltip("tasa_100k:Q", title="Tasa", format=".2f")
    ]
)

(barras + linea).resolve_scale(
    y="independent"
).properties(
    width=800,
    height=450,
    title="Agresiones (homicidios): defunciones y tasa por 100,000 habitantes"
)

alt.LayerChart(...)

In [27]:
# Menor diferencia (ascending)
bottom_30_diferencia = (
    diferencia_tasa
    .dropna(subset=["diferencia_tasa_100k"])
    .nsmallest(30, "diferencia_tasa_100k")
)

alt.Chart(bottom_30_diferencia).mark_bar().encode(
    x=alt.X(
        "diferencia_tasa_100k:Q",
        title="Diferencia de tasa por 100,000 habitantes",
        axis=alt.Axis(format=".2f")
    ),
    y=alt.Y(
        "lista_mex_desc:N",
        sort="x",
        title="Causa de defunción"
    ),
    tooltip=[
        alt.Tooltip("lista_mex_desc:N", title="Causa"),
        alt.Tooltip(
            "diferencia_tasa_100k:Q",
            title="Diferencia",
            format=".3f"
        )
    ]
).properties(
    width=500,
    height=700,
    title="Top 30 causas con menor diferencia en la tasa"
)

alt.Chart(...)